In [ ]:
import transformers
import torch
from datasets import load_dataset
import os 
import sys
from tqdm import tqdm

In [ ]:
# from huggingface_hub import login
# login()

## load MMMU dataset & extract keyword from [question, option] (https://huggingface.co/datasets/lmms-lab/MMMU)

In [ ]:
# data processing can learn from Cambrian (https://github.com/cambrian-mllm/cambrian/blob/main/eval/eval/mmmu/mmmu_eval.py)
validation_dataset = load_dataset("lmms-lab/MMMU", split="validation")
# dev_dataset = load_dataset("lmms-lab/MMMU", split="dev")
# test_dataset = load_dataset("lmms-lab/MMMU", split="test")




In [ ]:
llama_template_long = """
    I have the following multiple choice question and its options. Please give me the keywords that are present in this context and separate them with commas. Make sure to exclude the words "question" and "options" as keywords. Do not provide any additional explanation or commentary—only return the keywords.
For example:
Question: Here are facts for the Hudson Roofing Company for December. <image 1> Assuming no investments or withdrawals, what is the ending balance in the owners' capital account? Options: ['$63,020', '$58,410', '$71,320', '$77,490']
The 10 keywords are: Hudson Roofing Company, December, Ending balance, Owners' capital account, Investments, Withdrawals, $63,020, $58,410, $71,320, $77,490.
Now, here is the new query:
- {query}
Please give 10 keywords for the new query:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
"""

llama_template = """
Please give me 10 keywords that are present in this question-option context and separate them with commas.
Make sure you to only return the keywords and say nothing else.
Make sure to exclude the words "question" and "options" as keywords
I have the following multiple choice question and its options:
- {query}
"""

# llama_prompt = llama_template.format(
#         query=query
#     )

# print(llama_prompt)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time


model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side = "left")
tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]



In [ ]:
class PromptIterableDataset(torch.utils.data.IterableDataset):
    def __init__(self, ds, template):
        self.ds = ds
        self.template = template

    def __iter__(self):
        for sample in self.ds:
            query = 'question: ' + str(sample['question']) + ' options: ' + str(sample['options'])
            llama_prompt = self.template.format(
                query=query
            )
            messages = [
                {"role": "system", "content": "You are a helpful chatbot that assists users in generating keywords from conversations. You have been given a conversation and need to generate keywords from it."},
                {"role": "user", "content": llama_prompt },
            ]
            yield messages

class PromptDataset(torch.utils.data.Dataset):
    def __init__(self, ds, template):
        self.ds = ds
        self.template = template

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]
        query = 'question: ' + str(sample['question']) + ' options: ' + str(sample['options'])
        llama_prompt = self.template.format(
            query=query
        )
        messages = [
            {"role": "system", "content": "You are a helpful chatbot that assists users in generating keywords from conversations. You have been given a conversation and need to generate keywords from it."},
            {"role": "user", "content": llama_prompt },
        ]
        return messages
    
prompt_ds = PromptDataset(validation_dataset, llama_template) # we use the subset of the dataset with COCO captions

output_dir = '/home/jizej/Workspaces/UOUO/keyword/mmmu_gpt/val_keyword'
# create dir if not exists
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

prompt_loader = torch.utils.data.DataLoader(prompt_ds, batch_size=16, collate_fn=lambda x: [_ for _ in x], shuffle=False)

llama_keywords = []
for i, batch in enumerate(tqdm(prompt_loader)):
    texts = tokenizer.apply_chat_template(batch, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(texts, padding="longest", return_tensors="pt").to(model.device)
    # inputs = {key: val. for key, val in inputs.items()}
    temp_texts=tokenizer.batch_decode(inputs["input_ids"], skip_special_tokens=True)


    start_time = time.time()
    gen_tokens = model.generate(
        **inputs, 
        max_new_tokens=128, 
        pad_token_id=tokenizer.eos_token_id, 
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.6,
        top_p=0.9
    )

    gen_text = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
    gen_text = [i[len(temp_texts[idx]):] for idx, i in enumerate(gen_text)]

    llama_keywords.extend(gen_text)

    # write to file every 10 batches or at the end
    if i % 10 == 0 or i == len(prompt_loader) - 1:
        with open(os.path.join(output_dir, f"llama_keywords_{i}.txt"), "w") as f:
            f.write("\n".join(llama_keywords))
        llama_keywords = []

In [ ]:
torch.cuda.empty_cache()

In [ ]:
print(torch.cuda.memory_summary())

# Check total allocated memory on the current device
print(f"Allocated memory: {torch.cuda.memory_allocated() / 1024**3} GB")

# Check total reserved memory on the current device
print(f"Reserved memory: {torch.cuda.memory_reserved() / 1024**3} GB")